In [1]:
%load_ext dotenv
%dotenv .env.hf-token

In [20]:
PUSH = False
TASK = "tengeop"

In [21]:
import torch
import torchvision
from torchinfo import summary

with torch.no_grad():
    resnet50 = torchvision.models.resnet50(weights=None)
    resnet50.fc = torch.nn.Identity()
    
    conv1 = resnet50.conv1
    new_conv1 = torch.nn.Conv2d(
                in_channels=1,
                out_channels=conv1.out_channels,
                kernel_size=conv1.kernel_size,
                stride=conv1.stride,
                padding=conv1.padding,
                bias=(conv1.bias is not None)
            )
    
    new_conv1.weight = torch.nn.Parameter(conv1.weight.sum(dim=1, keepdim=True))

    resnet50.conv1 = new_conv1

summary(resnet50, (1, 1, 256, 256), depth=10, row_settings=["var_names"])

Layer (type (var_name))                  Output Shape              Param #
ResNet (ResNet)                          [1, 2048]                 --
├─Conv2d (conv1)                         [1, 64, 128, 128]         3,136
├─BatchNorm2d (bn1)                      [1, 64, 128, 128]         128
├─ReLU (relu)                            [1, 64, 128, 128]         --
├─MaxPool2d (maxpool)                    [1, 64, 64, 64]           --
├─Sequential (layer1)                    [1, 256, 64, 64]          --
│    └─Bottleneck (0)                    [1, 256, 64, 64]          --
│    │    └─Conv2d (conv1)               [1, 64, 64, 64]           4,096
│    │    └─BatchNorm2d (bn1)            [1, 64, 64, 64]           128
│    │    └─ReLU (relu)                  [1, 64, 64, 64]           --
│    │    └─Conv2d (conv2)               [1, 64, 64, 64]           36,864
│    │    └─BatchNorm2d (bn2)            [1, 64, 64, 64]           128
│    │    └─ReLU (relu)                  [1, 64, 64, 64]           --
│ 

In [22]:
PATH_RESNET_WEIGHTS = "resnet50.weights.pth.old"

In [23]:
import torch

state_dict_legacy = torch.load(PATH_RESNET_WEIGHTS, map_location="cpu")

In [24]:
state_dict_new = {}

trad = {
    "0": "conv1",
    "1": "bn1",
    "4": "layer1",
    "5": "layer2",
    "6": "layer3",
    "7": "layer4",
}

for k, v in state_dict_legacy.items():
    if k.startswith("teacher_backbone.backbone."):
        k_new = k.removeprefix("teacher_backbone.backbone.")
        for kk, vv in trad.items():
            if k_new.startswith(kk):
                k_new = k_new.replace(kk, vv, 1)
                break
        state_dict_new[k_new] = v

In [25]:
resnet50.load_state_dict(state_dict_new)

<All keys matched successfully>

In [22]:
import torch

resnet50.eval()
torch.save(resnet50.state_dict(), "resnet50.weights.pth")

In [23]:
from transformers import ResNetModel, ResNetConfig
from torchinfo import summary

config_resnet = ResNetConfig(1)
resnet50_hf = ResNetModel(config_resnet)

summary(resnet50_hf, (1, 1, 256, 256), row_settings=["var_names"], depth=10)

Layer (type (var_name))                                                Output Shape              Param #
ResNetModel (ResNetModel)                                              [1, 2048, 1, 1]           --
├─ResNetEmbeddings (embedder)                                          [1, 64, 64, 64]           --
│    └─ResNetConvLayer (embedder)                                      [1, 64, 128, 128]         --
│    │    └─Conv2d (convolution)                                       [1, 64, 128, 128]         3,136
│    │    └─BatchNorm2d (normalization)                                [1, 64, 128, 128]         128
│    │    └─ReLU (activation)                                          [1, 64, 128, 128]         --
│    └─MaxPool2d (pooler)                                              [1, 64, 64, 64]           --
├─ResNetEncoder (encoder)                                              [1, 2048, 8, 8]           --
│    └─ModuleList (stages)                                             --                  

In [24]:
import json

with open("layer_name_mapping.json", "r") as f:
    layer_name_mapping = json.load(f)

In [25]:
state_dict_torch = resnet50.state_dict()

mapped_state_dict = {}
for k, v in state_dict_torch.items():
    if k in layer_name_mapping:
        mapped_state_dict[layer_name_mapping[k]] = v

resnet50_hf.load_state_dict(mapped_state_dict)

<All keys matched successfully>

In [26]:
resnet50_hf.eval()
resnet50_hf.save_pretrained("resnet50_hf")

In [27]:
if PUSH:
    import os

    resnet50_hf.eval()
    resnet50_hf.push_to_hub("galeio-research/nereus-sar-1", private=True, token=os.environ['HF_TOKEN'])

In [28]:
import os
from transformers import AutoModel

resnet_from_hub = AutoModel.from_pretrained("galeio-research/nereus-sar-1", token=os.environ['HF_TOKEN'])

In [29]:
import torch

x = torch.randn(1, 1, 256, 256)

resnet50_hf.cuda()
resnet50_hf.eval()
with torch.no_grad():
    output = resnet50_hf.forward(x.to("cuda"))

resnet_from_hub.cuda()
resnet_from_hub.eval()
with torch.no_grad():
    output_from_hub = resnet_from_hub.forward(x.to("cuda"))
output_from_hub

torch.all(output_from_hub.pooler_output == output.pooler_output)

tensor(True, device='cuda:0')

In [26]:
PATH_HEAD_W = f"probing_head_resnet50_{TASK}/w.pt"
PATH_HEAD_B = f"probing_head_resnet50_{TASK}/b.pt"

In [27]:
import torch

w = torch.load(PATH_HEAD_W, weights_only=False)
b = torch.load(PATH_HEAD_B, weights_only=False)

w = torch.from_numpy(w)
b = torch.from_numpy(b)

if TASK == "wind" or TASK == "wave":
    w = w.unsqueeze(1)
    w = w.T

state_dict_head = {'weight': w, 'bias': b}

In [28]:
import torch

if TASK == "tengeop":
    out_features = 10
elif TASK == "wave" or TASK == "wind":
    out_features = 1
    
head = torch.nn.Linear(2048, out_features)
head.load_state_dict(state_dict_head)

<All keys matched successfully>

In [33]:
head_path = f"linear-head_resnet50_{TASK}.weights.pth"

torch.save(head.state_dict(), head_path)

In [34]:
if TASK == "wind":
    PROBLEM_TYPE = "regression"
    NUM_LABELS = 1
    LABEL2ID = {"wind_speed": 0}
    ID2LABEL = {0: "wind_speed"}
elif TASK == "wave":
    PROBLEM_TYPE = "regression"
    NUM_LABELS = 1
    LABEL2ID = {"wave_height": 0}
    ID2LABEL = {0: "wave_height"}
elif TASK == "tengeop":
    PROBLEM_TYPE = "multi_label_classification"
    NUM_LABELS = 10

In [35]:
PATH_RESNET_HF_WEIGHTS = "resnet50_hf"

In [36]:
import torch
from transformers import ResNetForImageClassification, ResNetConfig
from torchinfo import summary

config_resnet = ResNetConfig.from_pretrained(PATH_RESNET_HF_WEIGHTS)
config_with_linear_head = ResNetConfig(hidden_size=2048, finetuning_task=TASK, num_labels=NUM_LABELS, problem_type=PROBLEM_TYPE, **config_resnet.to_diff_dict())
nereus = ResNetForImageClassification(config_with_linear_head)

summary(nereus, input_size=(1, 1, 256, 256), row_settings=["var_names"], depth=10)

Layer (type (var_name))                                                     Output Shape              Param #
ResNetForImageClassification (ResNetForImageClassification)                 [1, 1]                    --
├─ResNetModel (resnet)                                                      [1, 2048, 1, 1]           --
│    └─ResNetEmbeddings (embedder)                                          [1, 64, 64, 64]           --
│    │    └─ResNetConvLayer (embedder)                                      [1, 64, 128, 128]         --
│    │    │    └─Conv2d (convolution)                                       [1, 64, 128, 128]         3,136
│    │    │    └─BatchNorm2d (normalization)                                [1, 64, 128, 128]         128
│    │    │    └─ReLU (activation)                                          [1, 64, 128, 128]         --
│    │    └─MaxPool2d (pooler)                                              [1, 64, 64, 64]           --
│    └─ResNetEncoder (encoder)                

In [37]:
PATH_RESNET_WEIGHTS_SAFE = "resnet50_hf/model.safetensors"
PATH_HEAD_WEIGHTS = f"linear-head_resnet50_{TASK}.weights.pth"

In [38]:
from safetensors.torch import load_file

state_dict_resnet = load_file(PATH_RESNET_WEIGHTS_SAFE)
nereus.resnet.load_state_dict(state_dict_resnet)

state_dict_head = torch.load(PATH_HEAD_WEIGHTS)
nereus.classifier[1].load_state_dict(state_dict_head)

<All keys matched successfully>

In [39]:
if PUSH:
    import os

    nereus.push_to_hub(f"galeio-research/nereus-sar-1-{TASK}",private=True, token=os.environ['HF_TOKEN'], generate_card=False)

In [40]:
import os
from transformers import AutoModelForImageClassification

nereus_from_hub = AutoModelForImageClassification.from_pretrained(f"galeio-research/nereus-sar-1-{TASK}", token=os.environ['HF_TOKEN']).to("cuda")

In [41]:
import torch

x = torch.randn(1, 1, 256, 256).cuda()

nereus.cuda()
nereus.eval()
with torch.no_grad():
    outputs = nereus.forward(x)

nereus_from_hub.cuda()
nereus_from_hub.eval()
with torch.no_grad():
    outputs_from_hub = nereus_from_hub(x)

torch.all(outputs_from_hub.logits == outputs.logits)

tensor(True, device='cuda:0')

In [35]:
from polis.datasets import Hdf5ImageDataset
import torchvision.transforms as transforms
from polis.transforms import LogImage

if TASK == "tengeop":
    class_field = "class"
    root = "/home/antoine/data/tengeop/tengeop_downsampled_small/test"
elif TASK == "wind":
    class_field = "wspd_era5"
    root = "/home/antoine/data/obukhov_downsampled_filtered_small/test"
elif TASK == "wave":
    class_field = "swh_alti"
    root = "/home/antoine/data/downsampled_images_small/test"

dataset = Hdf5ImageDataset(
    root=root,
    data_field="data",
    class_field=class_field,
    transform=transforms.Compose([
        transforms.CenterCrop((464, 416)),
        LogImage(),
        transforms.Normalize(mean=[5.667], std=[0.737]),
    ]),
    cache_metadata=False,
    unpad_images=True,
)

print(dataset)

Hdf5ImageDataset(root='/home/antoine/data/tengeop/tengeop_downsampled_small/test', data_field='data', class_field='class', shape=(1000, 1, 494, 574), min_dims=(475, 432), num_classes=10, unpad_images=True, transforms=True, target_transforms=False, )


In [43]:
MODEL_MODE = "train"

In [44]:
import os
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    f"galeio-research/nereus-sar-1-{TASK}", token=os.environ["HF_TOKEN"]
)
model.cuda()

if MODEL_MODE == "train":
    model.train()
else:
    model.eval()

loader = DataLoader(dataset, batch_size=128)

if TASK == "tengeop":
    n = 0
    for x, y in loader:
        x = x.to("cuda")
        with torch.no_grad():
            outputs = model(x)

        y_pred = torch.argmax(outputs.logits, dim=1)

        n += torch.sum(y_pred == y.to("cuda"))

    print("accuracy :", n / len(dataset))

elif TASK == "wave" or TASK == "wind":
    sqe = torch.tensor([]).to("cuda")

    for x, y in loader:
        with torch.no_grad():
            outputs = model(x.to("cuda"))

        y_pred = outputs.logits
        t = torch.tensor(list(dataset.classes[i] for i in y), dtype=torch.float32).to("cuda")

        diff = y_pred.flatten() - t

        sqe = torch.cat((sqe, torch.pow(diff, 2)))

    print("rmse :", torch.sqrt(sqe.mean()))


rmse : tensor(0.6987, device='cuda:0')


In [45]:
import torch

w = torch.load(PATH_HEAD_W, weights_only=False)
w = torch.from_numpy(w)
torch.all(w.to("cuda") == model.classifier[1].weight.to("cuda"))

tensor(True, device='cuda:0')

In [46]:
import torch

b = torch.load(PATH_HEAD_B, weights_only=False)
b = torch.from_numpy(b)
torch.all(b.to("cuda") == model.classifier[1].bias.to("cuda"))

tensor(True, device='cuda:0')

In [47]:
import os
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    f"galeio-research/nereus-sar-1-{TASK}", token=os.environ["HF_TOKEN"]
)

for key in mapped_state_dict:
    if key in model.resnet.state_dict():
        match = torch.all(mapped_state_dict[key].cpu() == model.resnet.state_dict()[key])
        print(f"{key}: {'Match' if match else 'Different'}")
    else:
        print(f"{key}: Not found in model.resnet state dict")


embedder.embedder.convolution.weight: Match
embedder.embedder.normalization.weight: Match
embedder.embedder.normalization.bias: Match
embedder.embedder.normalization.running_mean: Match
embedder.embedder.normalization.running_var: Match
encoder.stages.0.layers.0.layer.0.convolution.weight: Match
encoder.stages.0.layers.0.layer.0.normalization.weight: Match
encoder.stages.0.layers.0.layer.0.normalization.bias: Match
encoder.stages.0.layers.0.layer.0.normalization.running_mean: Match
encoder.stages.0.layers.0.layer.0.normalization.running_var: Match
encoder.stages.0.layers.0.layer.1.convolution.weight: Match
encoder.stages.0.layers.0.layer.1.normalization.weight: Match
encoder.stages.0.layers.0.layer.1.normalization.bias: Match
encoder.stages.0.layers.0.layer.1.normalization.running_mean: Match
encoder.stages.0.layers.0.layer.1.normalization.running_var: Match
encoder.stages.0.layers.0.layer.2.convolution.weight: Match
encoder.stages.0.layers.0.layer.2.normalization.weight: Match
encoder